# 05 · Compare exported models

Runs every bundle in `exports/` **through Core ML on the Mac** (the same converted models the
phone runs, not PyTorch) on the same test zip, and puts them side by side: mask quality,
distance and size error, false positives on background-only samples, latency and file size.

Needs macOS. Laptop and phone numbers can still differ a little (different chips, resizing);
the self-test on the phone measures that gap.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
EXPORTS = Path("../exports")

In [ ]:
import platform, tempfile
import numpy as np
from unetultra import metrics
from unetultra.calibration import LensCalibration
from unetultra.dataset import SessionDataset
from unetultra.export import coreml_predict, extract_package, read_card
from unetultra.session import find_sessions

assert platform.system() == "Darwin", "Core ML models only run on macOS"
import coremltools as ct

TEST_ZIPS = find_sessions(DATA, "test")
assert TEST_ZIPS, "no test session found in ../data"
calibration = LensCalibration.load(MODELS / "calibration.json")
test_set = SessionDataset(TEST_ZIPS, calibration)

results, summaries, extra = {}, {}, {}
for bundle in sorted(EXPORTS.glob("*.unetmodel")):
    card = read_card(bundle)
    with tempfile.TemporaryDirectory() as tmp:
        mlmodel = ct.models.MLModel(str(extract_package(bundle, tmp)))
        coreml_predict(mlmodel, np.zeros((card["input"]["height"], card["input"]["width"], 3), np.uint8),
                       np.zeros(len(card["meta_features"]), np.float32))   # warm-up, not timed
        rows, image_rows, latencies = metrics.evaluate_predictor(
            lambda image, meta: coreml_predict(mlmodel, image, meta), test_set, card["meta_features"])
    label = card["display_name"]
    results[label], summaries[label] = rows, metrics.summarise(rows, image_rows)
    extra[label] = {"latency_ms": float(np.median(latencies)), "size_mb": card["size_mb"],
                    "architecture": card["architecture"], "precision": card["precision"]}
    print("evaluated", label)

## Table

In [ ]:
print(metrics.format_summary(summaries))
print()
print(f"{'':38}" + "".join(f"{label:>18}" for label in extra))
print(f"{'Mac latency (median)':38}" + "".join(f"{e['latency_ms']:15.1f} ms" for e in extra.values()))
print(f"{'model size':38}" + "".join(f"{e['size_mb']:15.1f} MB" for e in extra.values()))
print(f"{'precision':38}" + "".join(f"{e['precision']:>18}" for e in extra.values()))

## Error against distance

In [ ]:
from unetultra import plots

plots.predicted_vs_true_distance(results)
plots.error_vs_distance(results)
plots.error_vs_distance(results, key="pred_width_cm", truth="true_width_cm", ylabel="width error (cm)");